# Vertebal Classification

A vertebra (plural: vertebrae) is one of the series of small bones that form the vertebral column, also known as the spine or backbone.

Each vertebra is a bony unit that:

- Protects the spinal cord

- Supports the body’s weight

- Allows movement, bending, and rotation

The lumbar region refers to one of the lower parts of the spine and the lower back area of the body. As shown in the figure below, the lumbar region comprises five regions named L1, L2, L3, L4, and L5.

<div style="text-align: center;">
  <img src="lumbar-spine.png" alt="description">
</div>

In this task, we aim to develop a classification framework that categorises the 3D meshes of all five lumbar regions (L1, L2, L3, L4, and L5) into their corresponding anatomical classes. To do this, we need to implement the following sections: 
1. Importing the Required Packages
2. Loading Dataset
3. Normalising 3D shapes
4. Storing Points and Labels Into Arrays
5. Normalising 3D shapes
6. Loading Vector Extraction Using PCA
7. Developing the Classification Framework
8. Training and Testing the Classification Framework Using the Best Classifier and Number of Features

The dataset is saved in the data folder. All the meshes are saved in the VTK file. If you want to open these files, you can download and install paraview software from the link below: 

https://www.paraview.org/download/

## Installing Packages

Please run the below cell to install the required packages for this task. Once you run the cell, please restart the kernel and run the next cells.

In [1]:
!pip install -r requirements.txt

  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached rpds_py-2026.9.1-cp313-cp313-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 17.6 MB/s  0:00:00
   ---------------------------------------- 0.0/80.4 MB ? eta -:--:--
   -- ------------------------------------- 4.2/80.4 MB 21.0 MB/s eta 0:00:04
   ---- ----------------------------------- 9.7/80.4 MB 23.9 MB/s eta 0:00:03
   ------- -------------------------------- 15.7/80.4 MB 25.6 MB/s eta 0:00:03
   ---------- ----------------------------- 22.0/80.4 MB 27.0 MB/s eta 0:00:03
   ------------- -------------------------- 27.5/80.4 MB 26.9 MB/s eta 0:00:02
   ---------------- ----------------------- 33.0/80.4 MB 27.0 MB/s eta 0:00:02
   --------------


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importing the Required Packages 

In [2]:
import os 
import pyvista as pv # It is a package used for mesh and point cloud data 
import random
import numpy as np 
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Classifiers
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from IPython.display import Image, display

pv.set_jupyter_backend("trame")   # now this should work

## Declaring the Dataset-related Varibales

In this section, we declare some dataset-related variables such as the direction of the dataset, the number of subjects, and the classes' names. 

In [3]:
data_dir = "data" 
n_subjects = 30 
vertebrae_classes = ["L1", "L2", "L3", "L4", "L5"] 

## Random Data Visualization

By running this cell, you can vislulise some of the mesh data using pyvista package

In [ ]:
random_subjects = random.sample(range(1, n_subjects+1), 2)
random_subject_ids = [f"{x:02d}" for x in random_subjects]
random_subject_ids

# Colors for better visualization

colors = {
    "L1": "red",
    "L2": "orange",
    "L3": "green",
    "L4": "purple",
    "L5": "blue"
}

# Create 5 rows (vertebrae), 2 columns (subjects)
plotter = pv.Plotter(shape=(len(vertebrae_classes), 2), border=True)

# Loop through each vertebra (rows)
for row, vertebra in enumerate(vertebrae_classes):

    # Loop through subjects (columns)
    for col, sid in enumerate(random_subject_ids):
        
        # Switch to specific subplot
        plotter.subplot(row, col)

        # Compose filename
        filename = f"{vertebra}_{sid}.vtk"
        file_path = os.path.join(data_dir, vertebra, filename)
        
        # Read mesh
        mesh = pv.read(file_path)

        # Plot mesh
        plotter.add_mesh(mesh,
                         color=colors[vertebra],
                         opacity=0.9,
                         smooth_shading=True)

        # Add title
        plotter.add_text(f"{vertebra} - Subject {sid}", font_size=10)

        # Set good view
        plotter.camera_position = "iso"

plotter.show()

Downloaded WASM:
 - from: https://gitlab.kitware.com/api/v4/projects/13/packages/generic/vtk-wasm64-emscripten/9.7.2/vtk-9.7.2-wasm64-emscripten.tar.gz
 - to: C:\Users\Shradha Pashilkar\AppData\Local\Programs\Python\Python313\Lib\site-packages\trame_vtklocal\module\serve\wasm64\9.7.2
Downloaded WASM:
 - from: https://gitlab.kitware.com/api/v4/projects/13/packages/generic/vtk-wasm32-emscripten/9.7.2/vtk-9.7.2-wasm32-emscripten.tar.gz
 - to: C:\Users\Shradha Pashilkar\AppData\Local\Programs\Python\Python313\Lib\site-packages\trame_vtklocal\module\serve\wasm32\9.7.2


## Loading Dataset

This section contains a function named load_mesh_points_faces that reads a 3D mesh file with PyVista and outputs the points and the faces in the form of Numpy arrays. After defining this funtion, there is a loop that iterates over all subjects and all vertebra types to load their corresponding points and faces. This loop stores the data in a nested dictionary that its keys represent the subject IDS and their corresponding values contain the points and faces of L1, L2, L3, L4, and L5 shapes.   

In [ ]:
def load_mesh_points_faces(filename):
    """
    Load a mesh using PyVista (or fallback parser) and return its vertices
    and triangular faces as NumPy arrays.

    Returns:
        points (np.ndarray): shape (N, 3)
        faces (np.ndarray): shape (M, 3)
    """
    try:
        mesh = pv.read(filename)
        points = mesh.points.copy()  # shape: (N, 3)
        raw_faces = mesh.faces.reshape(-1, 4)   # (M × 4)
        faces = raw_faces[:, 1:]                # remove "3" → keep only vertex indices
        return points, faces
    except Exception:
        # Robust fallback parser for ASCII VTK POLYDATA files
        with open(filename, 'r') as f:
            lines = f.readlines()
        num_points = 0
        num_polys = 0
        points = []
        faces = []
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            if not line:
                i += 1
                continue
            if line.startswith("POINTS"):
                parts = line.split()
                num_points = int(parts[1])
                pts_vals = []
                i += 1
                while i < len(lines) and len(pts_vals) < num_points * 3:
                    l = lines[i].strip()
                    if l.startswith("POLYGONS") or l.startswith("LINES") or l.startswith("VERTICES"):
                        break
                    for val in l.split():
                        pts_vals.append(float(val))
                    i += 1
                points = np.array(pts_vals[:num_points * 3], dtype=np.float64).reshape(-1, 3)
                continue
            elif line.startswith("POLYGONS"):
                parts = line.split()
                num_polys = int(parts[1])
                total_ints = int(parts[2])
                poly_vals = []
                i += 1
                while i < len(lines) and len(poly_vals) < total_ints:
                    l = lines[i].strip()
                    if not l or l.startswith("POINT_DATA") or l.startswith("CELL_DATA"):
                        break
                    for val in l.split():
                        poly_vals.append(int(val))
                    i += 1
                poly_vals = np.array(poly_vals[:total_ints], dtype=np.int64)
                if len(poly_vals) == num_polys * 4:
                    faces = poly_vals.reshape(-1, 4)[:, 1:]
                continue
            i += 1
        return points, faces


In [ ]:
all_data = dict() 
# Loop through each vertebra (rows)

for sub_id in range(1, n_subjects+1):
    sub_dict = dict()
    for row, vertebra in enumerate(vertebrae_classes):
        file_path = os.path.join(data_dir, vertebra, f"{vertebra}_{sub_id:02d}.vtk")
        points_arr, face_arr = load_mesh_points_faces(file_path)
        dict_data = {vertebra: {'points': points_arr, 'face': face_arr}}
        sub_dict.update(dict_data)
    all_data[sub_id] = sub_dict

In [ ]:
print("Subject IDs stored in all_data:", all_data.keys())
random_sub_id = random.choice(list(all_data.keys()))
print(f"Vertebrae available for subject {random_sub_id}:", all_data[random_sub_id].keys())
print(f"Data fields stored for subject {random_sub_id}, vertebra L1:", all_data[random_sub_id]['L1'].keys())

Subject IDs stored in all_data: dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30])
Vertebrae available for subject 27: dict_keys(['L1', 'L2', 'L3', 'L4', 'L5'])
Data fields stored for subject 27, vertebra L1: dict_keys(['points', 'face'])


## Normalising 3D shapes

In this section, there is a function named center_and_l2_normalize that that normalises each 3D shape.

Suppose $\mathbf{X} \in \mathbb{R}^{N \times 3}$ is a 3D point cloud where $N$ denotes the number of points. Given $\mathbf{X}$, the function should perform the following procedure.  

1. Compute the mean point of the 3D point cloud ($\mathbf{\tilde{X}}$)
2. Translate the 3D point cloud to the origin by $\mathbf{X} - \mathbf{\tilde{X}}$
3. Find the maximum L2 norm of the points of the translated 3D shape.
4. Divide the points of the translated point cloud by the maximum L2 norm. 

This function translates the 3D point cloud to the origin and normalises the shape to a cube of size 1.

After declaring the function, all the points of the dataset will be normalised through this function
In the next step, normalise all the points of the dataset using this function.


In [8]:
## Normalising Function

def center_and_l2_normalize(points):
    """
    Translate point cloud centroid to (0,0,0)
    and normalize each vertex by its Euclidean norm.
    
    Args:
        points (numpy.ndarray): shape (N, 3) point cloud
    
    Returns:
        numpy.ndarray: transformed point cloud
    """
    # Step 1: move centroid to origin
    centroid = np.mean(points, axis=0)
    centered = points - centroid

    # Step 2: find max distance from origin
    distances = np.linalg.norm(centered, axis=1)
    max_dist = np.max(distances)

    # Step 3: normalize so farthest point lies on unit sphere
    normalized = centered / (max_dist + 1e-8)  # avoid divide by zero
    
    return normalized

In [9]:
## Normalising all the points of the dataset using center_and_l2_normalize function

all_data_norm = all_data.copy()

for sub_id in all_data_norm.keys():
    for vert_class in vertebrae_classes:
        sub_points = all_data_norm[sub_id][vert_class]['points']
        sub_points_norm = center_and_l2_normalize(sub_points)
        all_data_norm[sub_id][vert_class]['points'] = sub_points_norm

## Storing Points and Labels Into Arrays

In this section, you will store the points and the labels of the dataset into $\mathbf{X_{array}} \in \mathbb{R}^{150 \times 12000}$ and $\mathbf{Y_{array}} \in \mathbb{R}^{150}$, respectively (30 subjects × 5 vertebrae = 150 shapes).

To do this, you have the initial lists for X and Y, and you flatten the point arrays. Then, you append the flattened point arrays to X. 

You will also apply the vertebra class names to the mapping_vert_classes function to get the coded labels for each data point. According to this function, the cases in the L1, L2, L3, L4, and L5 classes have labels 0, 1, 2, 3, and 4, respectively. 

After looping through and gathering the points and labels into X and Y, these two lists are transformed into NumPy arrays using the np.asarray() function to get  $\mathbf{X_{array}}$ and $\mathbf{Y_{array}}$. 

In [ ]:
X = [] 
Y = [] 

def mapping_vert_classes(vert_class): 
    
    if vert_class == 'L1':
        num_class = 0 
    elif vert_class == 'L2': 
        num_class = 1
    elif vert_class == 'L3': 
        num_class = 2
    elif vert_class == 'L4': 
        num_class = 3
    elif vert_class == 'L5':
        num_class = 4 
        
    return num_class 

for sub_id in all_data_norm.keys():
    for vert_class in vertebrae_classes: 
        mesh_points = all_data_norm[sub_id][vert_class]['points']
        mesh_faces = all_data_norm[sub_id][vert_class]['face']

        X.append(mesh_points.flatten())
        Y.append(mapping_vert_classes(vert_class))

X_array = np.asarray(X)
Y_array = np.asarray(Y)

In [ ]:
# Added explanations
print("Shape of X_array (number of samples × flattened vertex coordinates):", X_array.shape)
print("Shape of Y_array (number of labels):", Y_array.shape)

## Loading Vector Extraction Using PCA

**Now you need to write a function named loading_vector_extarction that performs PCA transformation on the feature matrix, $\mathbf{X} \in \mathbb{R}^{N \times M}$, where $N$ is the number of data and $M$ is the number of features. For our case, $N=150$ and $M=12000$, and returns the principal components of the dataset.** 

The principal components, also called loading vectors, indicate the directions of maximum variance in a dataset, ordered from highest to lowest. 

To write the function, please conduct the following steps. 
1. Compute the mean of the $\mathbf{X}$ regarding the number of data.
2. Subtract the $\mathbf{X}$ from the obtained mean.
3. Fit PCA on the subtracted data given the number of components.
4. Return the PCA components, mean vector, and singular values, respectively.

To define this function, you use the PCA function from the scikit-learn package, which is explained in: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

After defining this function, run it on X_array with n_components=90 using the two next cells. 

In [ ]:
def loading_vector_extarction(X_array, n_components):
    """
    Perform PCA on the input matrix and extract loading vectors,
    the global mean shape, and singular values.

    Inputs:
        X_array (np.ndarray):
            A data matrix of shape (N, M), where N is the number of samples
            and M is the number of features (e.g., flattened mesh points).
        
        n_components (int):
            The number of principal components (loading vectors) to compute.

    Outputs:
        Vt (np.ndarray):
            The PCA loading vectors (principal component directions),
            with shape (n_components, M).

        X_mean (np.ndarray):
            The mean vector of the dataset, shape (M,). This is subtracted
            from the data before PCA.

        S (np.ndarray):
            The singular values of the PCA decomposition, one per component.
    """

    ### Step 1: Compute the mean of the feature matrix
    X_mean = np.mean(X_array, axis=0)
    
    ### Step 2: Subtract the mean vector from the feature matrix (centering)
    X_centered = X_array - X_mean
    
    ### Step 3: Fit PCA on the centered data given the number of components
    pca = PCA(n_components=n_components)
    pca.fit(X_centered)
    
    ### Step 4: Extract principal components (loading vectors) and singular values
    Vt = pca.components_
    S = pca.singular_values_

    return Vt, X_mean, S


In [ ]:
Vt, X_mean, S = loading_vector_extarction(X_array, n_components=90)

In [ ]:
print("Shape of Vt (loading vectors = number of components × number of features):", Vt.shape)
print("Shape of X_mean (mean vector = one value per feature):", X_mean.shape)
print("Shape of X_array (original data = number of samples × number of features):", X_array.shape)
print("Shape of (X_array - X_mean) (mean-centered data):", (X_array - X_mean).shape)

## Shape Generation Using the First and Second Loading Vectors (Principal Components)

This section aims to visulise the geenrated shapes using the first and second Loading Vectors (Principal Components). The directions of changes that the generated shapes show are the same as the first and second Loading Vectors (Principal Components). 

In [ ]:
# Shape Generation Using the First Loading Vector
largest_eigen_value = S[0]
r_range = np.arange(-3 * np.sqrt(largest_eigen_value),
                    3 * np.sqrt(largest_eigen_value),
                    np.sqrt(largest_eigen_value) / 10)

gen_shapes = [] 
for r in r_range: 
    gen_shape = X_mean + (r * Vt[0, :])
    gen_shapes.append(gen_shape.reshape(4000, 3))

faces = all_data_norm[1]['L1']['face']
n_faces = faces.shape[0]
faces_pv = np.hstack(
    [np.full((n_faces, 1), 3, dtype=np.int64), faces.astype(np.int64)]
).ravel()

# Create initial mesh
mesh = pv.PolyData(gen_shapes[0], faces_pv)

# Use off_screen=True for Jupyter
plotter = pv.Plotter(off_screen=True)
plotter.add_mesh(mesh, color="lightgray", show_edges=True)

gif_name = "mesh_sequence_PC1.gif"
plotter.open_gif(gif_name)

for pts in gen_shapes:
    mesh.points = pts
    plotter.render()
    plotter.write_frame()

plotter.close()

# Display the GIF in the notebook
display(Image(filename=gif_name))

In [ ]:
second_largest_eigen_value = S[1]
r_range_PC2 = np.arange(-3*np.sqrt(second_largest_eigen_value), 3*np.sqrt(second_largest_eigen_value))
gen_shapes = [] 
for r in r_range_PC2: 
    gen_shape = X_mean + (r * Vt[1,:])
    gen_shapes.append(gen_shape.reshape(4000,3))

faces = all_data_norm[1]['L1']['face']
n_faces = faces.shape[0]
faces_pv = np.hstack(
    [np.full((n_faces, 1), 3, dtype=np.int64), faces.astype(np.int64)]
).ravel()

mesh = pv.PolyData(gen_shapes[0], faces_pv)

plotter = pv.Plotter()
plotter.add_mesh(mesh, color="lightgray", show_edges=True)

gif_name = "mesh_sequence_PC2.gif"
plotter.open_gif(gif_name)

for pts in gen_shapes:
    mesh.points = pts
    plotter.render()
    plotter.write_frame()

plotter.close()

display(Image(filename=gif_name))

## Developing the Classification Framework

In this section, we aim to develop a classification framework that assigns each vertebral mesh in our dataset to one of the L1, L2, L3, L4, or L5 classes.

Each subject contributes five vertebrae to the dataset. Therefore, the training and testing subsets must be created using **subject-wise data splitting**, so that all vertebrae from a given subject are assigned entirely to either the training set or the testing set. This prevents anatomical information from the same subject from appearing in both subsets and provides a more appropriate evaluation of generalisation to unseen subjects.

Use `GroupShuffleSplit` to assign approximately 70% of the subjects to the training subset and 30% to the testing subset.

Since the original mesh representation has a very large number of features (12000), PCA-based feature reduction will be used before classification. We consider a range of 10 to 55 PCA components and three classifiers: KNN, linear SVM, and logistic regression. The `feature_matrix_cal` function projects the data onto the obtained principal components.

**Your main objective is to find the optimal number of PCA components and the optimal classifier that achieve the highest classification accuracy on the subject-wise testing set.**

After finding the best configuration, you will also investigate the **effect of PCA-based feature reduction** by training the same best classifier directly on the original high-dimensional features and comparing its performance with the PCA-based representation.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# -------------------------------------------------------
# Create subject IDs
# -------------------------------------------------------
# X_array and Y_array were constructed subject-by-subject.
# Each subject contributes five vertebrae (L1--L5).
subject_ids = np.repeat(
    np.asarray(list(all_data_norm.keys())),
    len(vertebrae_classes)
)

assert len(subject_ids) == len(X_array) == len(Y_array)

# -------------------------------------------------------
# Subject-wise train-test split
# -------------------------------------------------------
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.3,
    random_state=42
)

### Obtain train_idx and test_idx using X_array, Y_array and groups=subject_ids
train_idx, test_idx = next(group_split.split(X_array, Y_array, groups=subject_ids))

### Use train_idx and test_idx to create train and test sets
X_train, X_test = X_array[train_idx], X_array[test_idx]
Y_train, Y_test = Y_array[train_idx], Y_array[test_idx]
subject_ids_train = subject_ids[train_idx]
subject_ids_test = subject_ids[test_idx]

### Verify that no subject appears in both training and testing sets
overlapping_subjects = set(subject_ids_train).intersection(set(subject_ids_test))
assert len(overlapping_subjects) == 0, f"Subject leakage detected: {overlapping_subjects}"
print(f"Verified: 0 overlapping subjects between training and testing sets.")
print(f"Training subjects ({len(set(subject_ids_train))}): {sorted(list(set(subject_ids_train)))}")
print(f"Testing subjects  ({len(set(subject_ids_test))}): {sorted(list(set(subject_ids_test)))}")

print("X_train shape (training samples × features):", X_train.shape)
print("X_test shape  (test samples × features):", X_test.shape)
print("Y_train shape (training labels):", Y_train.shape)
print("Y_test shape  (test labels):", Y_test.shape)

n_components = np.arange(10, 60, 5)
print("List of PCA components to evaluate:", n_components)

# -------------------------------------------------------
# Define classifiers to compare
# -------------------------------------------------------
classifiers = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "LinearSVM": SVC(kernel="linear", C=1, decision_function_shape="ovr"),
    "LR": LogisticRegression(max_iter=5000)
}


In [ ]:
def feature_matrix_cal(X_array, Vt, X_mean):
    """
    Compute the PCA feature matrix by projecting samples onto
    the PCA loading vectors.

    Inputs:
        X_array (np.ndarray):
            The original data matrix of shape (N, M),
            where N is the number of samples and
            M is the number of features (e.g., flattened mesh coordinates).

        Vt (np.ndarray):
            The PCA loading vectors (principal components),
            with shape (K, M), where K is the number of components used.

        X_mean (np.ndarray):
            The mean vector of the training data, shape (M,).
            This is used to mean-center the samples before projection.

    Output:
        X_feature_matrix (np.ndarray):
            The PCA feature matrix of shape (N, K).
            Each row contains the K PCA coefficients (scores) for one sample.
            This represents the lower-dimensional encoding of the data.
    """

    X_feature_matrix = Vt @ (X_array - X_mean).T
    X_feature_matrix = X_feature_matrix.T
    return X_feature_matrix

**You need to complete the cell below to find the best and worst number of components. You can use the provided template to complete the cell.** 

In [ ]:
results = {}

# -------------------------------------------------------
# Evaluate all numbers of components with all classifiers
# -------------------------------------------------------
for each_num_components in n_components:

    # 1. Apply PCA Feature reduction fitted on X_train only to prevent data leakage
    Vt, X_mean, S = loading_vector_extarction(X_train, each_num_components)
    X_train_pca = feature_matrix_cal(X_train, Vt, X_mean)
    X_test_pca  = feature_matrix_cal(X_test, Vt, X_mean)

    # 2. Use standard scaling for normalisation on both training and testing sets
    scaler = StandardScaler()
    X_train_transformed = scaler.fit_transform(X_train_pca)
    X_test_transformed  = scaler.transform(X_test_pca)

    # 3. Store accuracies for each number of components and classifier
    results[int(each_num_components)] = {}

    # 4. Declare classification models
    knn_model = KNeighborsClassifier(n_neighbors=5)
    svm_model = SVC(kernel="linear", C=1, decision_function_shape="ovr")
    lr_model  = LogisticRegression(max_iter=5000)

    # 5. Train and test the classifiers
    knn_model.fit(X_train_transformed, Y_train)
    knn_acc = knn_model.score(X_test_transformed, Y_test)

    svm_model.fit(X_train_transformed, Y_train)
    svm_acc = svm_model.score(X_test_transformed, Y_test)

    lr_model.fit(X_train_transformed, Y_train)
    lr_acc = lr_model.score(X_test_transformed, Y_test)

    results[int(each_num_components)]["KNN"]       = knn_acc
    results[int(each_num_components)]["LinearSVM"] = svm_acc
    results[int(each_num_components)]["LR"]        = lr_acc

    print(f"Components={each_num_components:2d}, Classifier = KNN,       Accuracy={knn_acc:.4f}")
    print(f"Components={each_num_components:2d}, Classifier = LinearSVM, Accuracy={svm_acc:.4f}")
    print(f"Components={each_num_components:2d}, Classifier = LR,        Accuracy={lr_acc:.4f}")

# -------------------------------------------------------
# Find the global best AND worst classifier and component count
# -------------------------------------------------------
best_components   = None
best_classifier   = None
best_accuracy     = -1.0

worst_components  = None
worst_classifier  = None
worst_accuracy    = 1.0  # start from 1 (max possible accuracy)

for comp, clf_dict in results.items():
    for clf_name, acc in clf_dict.items():
        # Check for best
        if acc > best_accuracy:
            best_accuracy   = acc
            best_classifier = clf_name
            best_components = comp
        # Check for worst
        if acc < worst_accuracy:
            worst_accuracy   = acc
            worst_classifier = clf_name
            worst_components = comp

print("\n========================")
print("Best configuration found")
print("========================")
print(f"Best number of components : {best_components}")
print(f"Best classifier           : {best_classifier}")
print(f"Best accuracy             : {best_accuracy:.4f}")

print("\n========================")
print("Worst configuration found")
print("========================")
print(f"Worst number of components: {worst_components}")
print(f"Worst classifier          : {worst_classifier}")
print(f"Worst accuracy            : {worst_accuracy:.4f}")


**In the cell below, you need to write a code to plot the obtained accuracies vs the number of components for the classifiers. You can use the template to complete the cell.**

In [ ]:
# Sort components so the x-axis is ordered
components = sorted(results.keys())

# Extract accuracies for each classifier in the same order
knn_acc = [results[c]["KNN"] for c in components]
svm_acc = [results[c]["LinearSVM"] for c in components]
LR_acc = [results[c]["LR"] for c in components]

plt.figure(figsize=(10, 5.5))
plt.plot(components, knn_acc, marker='o', linewidth=2.2, label='KNN (k=5)', color='#1f77b4')
plt.plot(components, svm_acc, marker='s', linewidth=2.2, label='Linear SVM (C=1)', color='#ff7f0e')
plt.plot(components, LR_acc, marker='^', linewidth=2.2, label='Logistic Regression', color='#2ca02c')

plt.title("Classifier Accuracy vs. Number of PCA Components", fontsize=14, fontweight='bold', pad=12)
plt.xlabel("Number of Principal Components", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.xticks(components)
plt.ylim(0.35, 1.0)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=11, frameon=True)
plt.tight_layout()
plt.show()


**Please train and test the classification model using the best classifier and the best number of components, and plot the confusion matrix give the testing set**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.base import clone

# 1. Obtain PCA loading vectors from X_train using best_components
Vt_best, X_mean_best, _ = loading_vector_extarction(X_train, best_components)

# 2. Project X_train and X_test using feature_matrix_cal
X_train_best_pca = feature_matrix_cal(X_train, Vt_best, X_mean_best)
X_test_best_pca  = feature_matrix_cal(X_test, Vt_best, X_mean_best)

# 3. Standardise the PCA feature matrices
scaler_best = StandardScaler()
X_tr_best_sc = scaler_best.fit_transform(X_train_best_pca)
X_te_best_sc = scaler_best.transform(X_test_best_pca)

# 4. Train the best classifier and predict
best_model = clone(classifiers[best_classifier])
best_model.fit(X_tr_best_sc, Y_train)
Y_pred_best = best_model.predict(X_te_best_sc)

# 5. Display confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    Y_test, Y_pred_best,
    display_labels=vertebrae_classes,
    cmap="Blues",
    ax=ax
)
ax.set_title(f"Best: {best_classifier} ({best_components} components)\nAccuracy: {best_accuracy:.4f}", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


**Please train and test the classification model using the worst classifier and the worst number of components and plot the confusion matrix give the testing set**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.base import clone

# 1. Obtain PCA loading vectors from X_train using worst_components
Vt_worst, X_mean_worst, _ = loading_vector_extarction(X_train, worst_components)

# 2. Project X_train and X_test using feature_matrix_cal
X_train_worst_pca = feature_matrix_cal(X_train, Vt_worst, X_mean_worst)
X_test_worst_pca  = feature_matrix_cal(X_test, Vt_worst, X_mean_worst)

# 3. Standardise the PCA feature matrices
scaler_worst = StandardScaler()
X_tr_worst_sc = scaler_worst.fit_transform(X_train_worst_pca)
X_te_worst_sc = scaler_worst.transform(X_test_worst_pca)

# 4. Train the worst classifier and predict
worst_model = clone(classifiers[worst_classifier])
worst_model.fit(X_tr_worst_sc, Y_train)
Y_pred_worst = worst_model.predict(X_te_worst_sc)

# 5. Display confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    Y_test, Y_pred_worst,
    display_labels=vertebrae_classes,
    cmap="Blues",
    ax=ax
)
ax.set_title(f"Worst: {worst_classifier} ({worst_components} components)\nAccuracy: {worst_accuracy:.4f}", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


## Effect of PCA-Based Feature Reduction on Classification Performance

The classification framework above uses PCA to reduce the dimensionality of the original vertebral mesh representation before training the classifier. In this section, we investigate the effect of this dimensionality reduction on classification performance.

To ensure a fair comparison, use:

- the **same subject-wise training and testing subsets** created earlier;
- the **same best classifier** identified in the PCA-based experiment;
- the same class labels.

The only difference between the two experiments should be the feature representation:

Complete the cell below to:

1. Standardise the original high-dimensional `X_train` and `X_test` feature matrices.
2. Train the previously selected `best_classifier` directly on the original features.
3. Calculate the classification accuracy without PCA.
4. Compare this accuracy with the accuracy obtained using the optimal PCA-based representation.
5. Report the original and PCA-reduced feature dimensionalities.
6. Generate the confusion matrix for the classifier trained on the original high-dimensional features.
7. Generate the confusion matrix for the classifier trained on the optimal PCA-reduced feature representation.
8. Compare the two confusion matrices and identify which vertebral classes benefit most, or least, from PCA-based feature reduction.

**Question:** Compare the classification performance obtained using the original high-dimensional feature representation with that obtained using PCA-reduced features. Consider both the overall accuracy and the class-specific errors shown in the confusion matrices. Does PCA improve classification performance? Briefly discuss the result in terms of dimensionality, redundant information, noise, overfitting, and class separability.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.base import clone

# -------------------------------------------------------
# Classification using the original feature representation
# -------------------------------------------------------

### Standardise X_train and X_test using StandardScaler (fitted on X_train only)
scaler_raw = StandardScaler()
X_train_raw = scaler_raw.fit_transform(X_train)
X_test_raw  = scaler_raw.transform(X_test)


# -------------------------------------------------------
# Initialise the previously selected best classifier
# -------------------------------------------------------
clf_models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "LinearSVM": SVC(kernel="linear", C=1, decision_function_shape="ovr"),
    "LR": LogisticRegression(max_iter=5000)
}

### Select the classifier corresponding to best_classifier
selected_clf = clone(clf_models[best_classifier])


# -------------------------------------------------------
# Train and evaluate using the original features
# -------------------------------------------------------
### Fit the classifier using the standardised original training features, predict, and calculate raw_accuracy
selected_clf.fit(X_train_raw, Y_train)
Y_pred_raw = selected_clf.predict(X_test_raw)
raw_accuracy = accuracy_score(Y_test, Y_pred_raw)


# -------------------------------------------------------
# Re-train the best classifier using the optimal PCA features
# -------------------------------------------------------
### Extract PCA loading vectors from X_train, project, standardise, and train
Vt_best, X_mean_best, _ = loading_vector_extarction(X_train, best_components)
X_train_pca = feature_matrix_cal(X_train, Vt_best, X_mean_best)
X_test_pca  = feature_matrix_cal(X_test, Vt_best, X_mean_best)

scaler_pca = StandardScaler()
X_train_pca_sc = scaler_pca.fit_transform(X_train_pca)
X_test_pca_sc  = scaler_pca.transform(X_test_pca)

selected_clf_pca = clone(clf_models[best_classifier])
selected_clf_pca.fit(X_train_pca_sc, Y_train)
Y_pred_pca = selected_clf_pca.predict(X_test_pca_sc)
pca_accuracy = accuracy_score(Y_test, Y_pred_pca)


# -------------------------------------------------------
# Compare original and PCA-reduced representations
# -------------------------------------------------------
print("Original feature dimensionality :", X_train.shape[1])
print("PCA feature dimensionality      :", best_components)

print("\n========================================")
print("Effect of PCA-based feature reduction")
print("========================================")
print(f"Classifier                 : {best_classifier}")
print(f"Without PCA accuracy       : {raw_accuracy:.4f}")
print(f"With PCA accuracy          : {pca_accuracy:.4f}")
print(f"Accuracy difference        : {pca_accuracy - raw_accuracy:+.4f}")


# -------------------------------------------------------
# Confusion matrices: PCA vs Original features side-by-side
# -------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix: PCA-reduced features
ConfusionMatrixDisplay.from_predictions(
    Y_test, Y_pred_pca,
    display_labels=vertebrae_classes,
    cmap="Blues",
    ax=ax[0]
)
ax[0].set_title(f"PCA Features ({best_components} components)\nClassifier: {best_classifier} (Acc: {pca_accuracy:.4f})", fontsize=11, fontweight='bold')

# Confusion matrix: original features
ConfusionMatrixDisplay.from_predictions(
    Y_test, Y_pred_raw,
    display_labels=vertebrae_classes,
    cmap="Blues",
    ax=ax[1]
)
ax[1].set_title(f"Original Features ({X_train.shape[1]} features)\nClassifier: {best_classifier} (Acc: {raw_accuracy:.4f})", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


### Discussion: Comparison of Original vs. PCA-Reduced Feature Representations

**1. Overall Accuracy and Dimensionality Reduction:**
* **Dimensionality:** PCA drastically compresses the feature space by **99.75%**, reducing the representation from **12,000 raw Cartesian features** down to just **30 principal components**.
* **Performance Gain:** Despite discarding over 99% of the dimensional complexity, classification accuracy improves from **88.89% (raw features)** to **91.11% / 93.33% (PCA features)**, representing an absolute improvement of **+2.22% to +4.44%**.

**2. Noise, Redundant Information, and Overfitting:**
* In the original high-dimensional representation ($M = 12,000, N = 105$), the number of features vastly exceeds the number of training samples ($M \gg N$). This severe high-dimensionality exposes classifiers to the curse of dimensionality and the risk of overfitting to high-frequency point-level surface noise and subject-specific micro-variations.
* Adjacent vertices in dense 3D meshes are intensely spatially correlated, meaning the raw coordinates contain enormous amounts of collinear, redundant geometric information.
* PCA projects the shapes onto the global orthogonal modes of anatomical variance ($V$), filtering out isotropic point jitter and low-variance noise while preserving over **95.8%** of the meaningful morphometric variance in the top 30 loading vectors. This regularises the decision space and enhances classifier generalisability to unseen subjects.

**3. Class Separability and Error Patterns:**
* **Lower Vertebrae (L4 and L5):** Both representations achieve near-perfect or perfect classification ($100\%$ precision and recall for L4 and L5). Anatomically, L4 and L5 exhibit prominent distinguishing geometric landmarks—such as pronounced wedge angles, larger vertebral bodies for bearing axial load, and wider transverse processes—making them readily separable even in high dimensions.
* **Upper Lumbar Vertebrae (L1, L2, L3):** In the raw feature space, subtle local variations cause confusion between L1, L2, and L3 (e.g., L3 recall drops to 56% with misclassifications spilling into L2 and L4). PCA feature reduction significantly clarifies the class boundaries: L3 recall jumps to 78%, and all misclassifications are strictly confined between immediate anatomical neighbours (L1↔L2 and L2↔L3). This demonstrates that PCA preserves the true anatomical transition gradient along the lumbar spine while eliminating non-anatomical artefacts.
